In [1]:
# Revised Jupyter Notebook cell: compile_layout_manifest_to_svg (robust overlap handling)

import json
import os
import re
from pathlib import Path
from xml.etree import ElementTree as ET
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from itertools import combinations
import shutil

try:
    from PIL import Image
except ImportError:
    !pip install Pillow -q
    from PIL import Image

# ------------------------------------------------------------
# 1. Asset pipeline: convert PNG/JPEG to WebP (85% reduction)
# ------------------------------------------------------------
def convert_new_assets_to_webp(assets_dir: str = "assets"):
    """Convert any .png/.jpg/.jpeg file in assets/ to .webp, updating manifest references."""
    assets_path = Path(assets_dir)
    if not assets_path.exists():
        print(f"Assets directory '{assets_dir}' not found. Creating it.")
        assets_path.mkdir(parents=True, exist_ok=True)
        return

    converted = []
    for ext in ("*.png", "*.jpg", "*.jpeg"):
        for img_file in assets_path.glob(ext):
            webp_file = img_file.with_suffix(".webp")
            if webp_file.exists():
                # Skip if webp already exists and newer than source
                if webp_file.stat().st_mtime >= img_file.stat().st_mtime:
                    continue
            # Convert
            try:
                with Image.open(img_file) as im:
                    im = im.convert("RGB")
                    im.save(webp_file, "webp", quality=85, optimize=True)
                converted.append(str(img_file))
                print(f"Converted: {img_file} → {webp_file}")
            except Exception as e:
                print(f"Error converting {img_file}: {e}")
    return converted

def update_manifest_image_hrefs(manifest: dict):
    """Replace .png/.jpg references in elements with .webp if file exists."""
    for elem in manifest.get("elements", []):
        if "style_attrs" in elem and "href" in elem.get("style_attrs", ""):
            match = re.search(r'href="(.*?)"', elem["style_attrs"])
            if match:
                href = match.group(1)
                if href.lower().endswith((".png", ".jpg", ".jpeg")):
                    webp_href = str(Path(href).with_suffix(".webp"))
                    elem["style_attrs"] = elem["style_attrs"].replace(href, webp_href)
                    print(f"Updated href: {href} → {webp_href}")

# ------------------------------------------------------------
# 2. Bounding box calculation
# ------------------------------------------------------------
@dataclass
class BBox:
    x: float
    y: float
    w: float
    h: float

    def area(self) -> float:
        return self.w * self.h

    def intersection(self, other: 'BBox') -> float:
        x_overlap = max(0, min(self.x + self.w, other.x + other.w) - max(self.x, other.x))
        y_overlap = max(0, min(self.y + self.h, other.y + other.h) - max(self.y, other.y))
        return x_overlap * y_overlap

    def contains(self, other: 'BBox', margin: float = 2.0) -> bool:
        """Return True if other bbox is fully inside this one (with optional margin tolerance)."""
        return (self.x - margin <= other.x and
                self.y - margin <= other.y and
                self.x + self.w + margin >= other.x + other.w and
                self.y + self.h + margin >= other.y + other.h)

    def overlap_ratio(self, other: 'BBox') -> float:
        inter = self.intersection(other)
        if inter == 0:
            return 0.0
        min_area = min(self.area(), other.area())
        if min_area == 0:
            return 0.0
        return inter / min_area

def get_element_bbox(elem: dict) -> Optional[BBox]:
    """Compute bounding box from geometry attributes. Simple heuristics for text."""
    obj_type = elem.get("object_type")
    geo = elem.get("geometry_attrs", {})

    if obj_type == "rect":
        return BBox(geo["x"], geo["y"], geo["width"], geo["height"])
    elif obj_type == "circle":
        cx, cy, r = geo["cx"], geo["cy"], geo["r"]
        return BBox(cx - r, cy - r, 2 * r, 2 * r)
    elif obj_type == "line":
        x1, y1, x2, y2 = geo["x1"], geo["y1"], geo["x2"], geo["y2"]
        return BBox(min(x1, x2), min(y1, y2), abs(x2 - x1), abs(y2 - y1))
    elif obj_type == "polygon":
        points_str = geo.get("points", "")
        coords = list(map(float, points_str.replace(",", " ").split()))
        xs = coords[0::2]
        ys = coords[1::2]
        return BBox(min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys))
    elif obj_type == "text":
        x = float(geo.get("x", 0))
        y = float(geo.get("y", 0))
        text = elem.get("content", "")
        style = elem.get("style_attrs", "")
        font_size = 16
        match = re.search(r'font-size:(\d+)px', style)
        if match:
            font_size = int(match.group(1))
        char_width = 0.6 * font_size
        lines = text.split("\n")
        max_line_width = max(len(line) * char_width for line in lines) if lines else 0
        text_height = font_size * len(lines)
        anchor = geo.get("text-anchor", "start")
        if anchor == "middle":
            x = x - max_line_width / 2
        elif anchor == "end":
            x = x - max_line_width
        baseline = geo.get("dominant-baseline", "alphabetic")
        if baseline == "middle":
            y = y - text_height / 2
        elif baseline in ("alphabetic", "auto"):
            y = y - text_height  # approximate top
        return BBox(x, y, max_line_width, text_height)
    else:
        return None

# ------------------------------------------------------------
# 3. Overlap detection – smart filtering
# ------------------------------------------------------------
SMALL_OVERLAP_AREA_THRESHOLD = 400  # px² – ignore tiny overlaps (e.g. arrowheads)

def is_problematic_overlap(elem1: dict, bb1: BBox, elem2: dict, bb2: BBox) -> bool:
    """Return True if the overlap between elem1 and elem2 is not intentional."""
    # Exclude background rectangle completely
    if elem1["object_id"] == "bg_rect" or elem2["object_id"] == "bg_rect":
        return False

    # If the smaller bbox is completely inside the larger, treat as intentional (text in box)
    if bb1.contains(bb2) or bb2.contains(bb1):
        return False

    # Compute overlap ratio and absolute area
    inter = bb1.intersection(bb2)
    if inter < SMALL_OVERLAP_AREA_THRESHOLD:
        return False

    ratio = bb1.overlap_ratio(bb2)
    return ratio > 0.05

def detect_overlaps(elements: List[dict]) -> List[str]:
    """Return a list of overlap descriptions for pairs that are truly problematic."""
    bboxes = {}
    for elem in elements:
        bb = get_element_bbox(elem)
        if bb:
            bboxes[elem["object_id"]] = (elem, bb)
        else:
            print(f"Warning: could not compute bbox for {elem['object_id']} ({elem['object_type']})")

    warnings = []
    for id1, id2 in combinations(bboxes.keys(), 2):
        elem1, bb1 = bboxes[id1]
        elem2, bb2 = bboxes[id2]
        if is_problematic_overlap(elem1, bb1, elem2, bb2):
            ratio = bb1.overlap_ratio(bb2)
            warnings.append(f"  - {id1} vs {id2}: {ratio*100:.1f}% overlap")
    return warnings

def generate_overlap_error_string(elements: List[dict]) -> str:
    """AI‑readable error string for remaining overlaps."""
    warnings = detect_overlaps(elements)
    if not warnings:
        return ""
    return "LAYOUT OVERLAP WARNING (some elements exceed safe overlap):\n" + "\n".join(warnings)

# ------------------------------------------------------------
# 4. SVG compilation
# ------------------------------------------------------------
def inject_manifest_into_template(template_path: str, manifest: dict) -> str:
    """Read background SVG and inject manifest elements before </svg>."""
    with open(template_path, "r", encoding="utf-8") as f:
        svg_string = f.read()

    if "</svg>" not in svg_string:
        raise ValueError("Template SVG missing closing </svg> tag")

    new_elements = []
    for elem in manifest["elements"]:
        obj_type = elem["object_type"]
        geo = elem["geometry_attrs"]
        style = elem.get("style_attrs", "")
        content = elem.get("content", "")
        z = elem.get("z_order", 0)

        if obj_type == "rect":
            el = f'<rect x="{geo["x"]}" y="{geo["y"]}" width="{geo["width"]}" height="{geo["height"]}" rx="{geo.get("rx",0)}" ry="{geo.get("ry",0)}" style="{style}" />'
        elif obj_type == "circle":
            el = f'<circle cx="{geo["cx"]}" cy="{geo["cy"]}" r="{geo["r"]}" style="{style}" />'
        elif obj_type == "line":
            el = f'<line x1="{geo["x1"]}" y1="{geo["y1"]}" x2="{geo["x2"]}" y2="{geo["y2"]}" style="{style}" />'
        elif obj_type == "polygon":
            el = f'<polygon points="{geo["points"]}" style="{style}" />'
        elif obj_type == "text":
            text_anchor = geo.get("text-anchor", "start")
            baseline = geo.get("dominant-baseline", "auto")
            el = f'<text x="{geo["x"]}" y="{geo["y"]}" text-anchor="{text_anchor}" dominant-baseline="{baseline}" style="{style}">{content}</text>'
        elif obj_type == "image":
            match = re.search(r'href=["\'](.*?)["\']', style)
            href = match.group(1) if match else ""
            el = f'<image x="{geo["x"]}" y="{geo["y"]}" width="{geo["width"]}" height="{geo["height"]}" href="{href}" style="{style}" preserveAspectRatio="xMidYMid meet" />'
        else:
            print(f"Unknown object_type: {obj_type}, skipping {elem['object_id']}")
            continue
        new_elements.append((z, el))

    new_elements.sort(key=lambda t: t[0])
    new_svg_lines = [el for _, el in new_elements]
    injection = "\n  <!-- Injected from layout_manifest.json -->\n  " + "\n  ".join(new_svg_lines)
    modified_svg = svg_string.replace("</svg>", injection + "\n</svg>")
    return modified_svg

# ------------------------------------------------------------
# 5. Main execution
# ------------------------------------------------------------
def main():
    # Step 0: Convert assets (if any)
    convert_new_assets_to_webp("assets")

    # Load manifest
    manifest_path = "layout_manifest.json"
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    update_manifest_image_hrefs(manifest)

    elements = manifest.get("elements", [])

    # Overlap analysis (warnings only, no abort)
    error_msg = generate_overlap_error_string(elements)
    if error_msg:
        print("\n" + error_msg)
        print("(SVG will still be generated; consider fine‑tuning positions.)")
    else:
        print("No problematic overlaps detected. Layout is clean.")

    # Compile SVG
    template = "pathoSlideTemp.svg"
    if not os.path.exists(template):
        raise FileNotFoundError(f"Template file '{template}' not found.")

    output_svg = inject_manifest_into_template(template, manifest)

    slide_id = manifest.get("slide_id", "unknown_slide")
    output_filename = f"final_slide_output_{slide_id}.svg"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(output_svg)

    print(f"Successfully generated: {output_filename}")

    # Bounding box summary (informational)
    print("\nBounding Box Summary:")
    for elem in elements:
        bbox = get_element_bbox(elem)
        if bbox:
            print(f"  {elem['object_id']:30s} -> x={bbox.x:7.1f} y={bbox.y:7.1f} w={bbox.w:6.1f} h={bbox.h:6.1f}")
        else:
            print(f"  {elem['object_id']:30s} -> [No BBox]")

if __name__ == "__main__":
    main()


LAYOUT OVERLAP WARNING (some elements exceed safe overlap):
  - node_restore_rect vs node_restore_text: 83.3% overlap
  - diamond_duration vs diamond_text: 87.7% overlap
  - box_path1_rect vs box_path1_text_restoration: 93.8% overlap
(SVG will still be generated; consider fine‑tuning positions.)
Successfully generated: final_slide_output_Slide20_IschaemiaReperfusion.svg

Bounding Box Summary:
  bg_rect                        -> x=    0.0 y=    0.0 w=1920.0 h= 955.0
  title                          -> x=  444.0 y=  110.0 w=1032.0 h=  40.0
  subtitle                       -> x=  625.2 y=  172.0 w= 669.6 h=  18.0
  node_restore_rect              -> x=  810.0 y=  220.0 w= 300.0 h=  60.0
  node_restore_text              -> x=  780.0 y=  238.0 w= 360.0 h=  24.0
  arrow_restore_to_duration_line -> x=  960.0 y=  280.0 w=   0.0 h=  35.0
  arrowhead_restore_to_duration  -> x=  955.0 y=  315.0 w=  10.0 h=   5.0
  diamond_duration               -> x=  860.0 y=  330.0 w= 200.0 h= 100.0
  diamond_t